<a href="https://colab.research.google.com/github/vunhankhanhmcs3-cmd/master-thesis-mooc-recommendation/blob/main/02_KG_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import pickle
from sklearn.model_selection import train_test_split
from google.colab import drive

In [ ]:
# Cấu hình
drive.mount('/content/drive')

# Đường dẫn (Cần đảm bảo bạn đã upload file raw vào đây)
BASE_PATH = '/content/drive/MyDrive/01. THESIS/My suggestion/'
PROCESSED_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/processed/')
RELATIONS_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/raw/relations/')
if not os.path.exists(PROCESSED_PATH): os.makedirs(PROCESSED_PATH)

Mounted at /content/drive


In [ ]:
# 1. Load dữ liệu từ File 1 (01_Data_Preprocessing.ipynb)
df_interact = pd.read_csv(os.path.join(PROCESSED_PATH, 'interactions_final.csv'))
with open(os.path.join(PROCESSED_PATH, 'course_map.pkl'), 'rb') as f:
    valid_course_map = pickle.load(f)
    valid_course_set = set(valid_course_map.keys())

num_users = df_interact['uid'].max() + 1
num_courses = df_interact['cid'].max() + 1

In [ ]:
# 2. CHIA TRAIN / VALIDATION / TEST (8:1:1)

RANDOM_STATE = 42

train_list, val_list, test_list = [], [], []

for uid, group in df_interact.groupby('uid'):
    if len(group) < 5:
        # Không đủ dữ liệu để chia 3 phần -> giữ hết cho train
        train_list.append(group)
        continue

    # Bước 1: 80% train / 20% temp
    tr, temp = train_test_split(
        group, test_size=0.2, random_state=RANDOM_STATE
    )

    # Bước 2: chia đôi temp -> 10% val / 10% test
    va, te = train_test_split(
        temp, test_size=0.5, random_state=RANDOM_STATE
    )

    train_list.append(tr)
    val_list.append(va)
    test_list.append(te)

train_df = pd.concat(train_list)
val_df = pd.concat(val_list)
test_df = pd.concat(test_list)

print(f"📊 Tổng số dòng: {len(df_interact)}")
print(f"   - Train: {len(train_df)} ({len(train_df)/len(df_interact)*100:.1f}%)")
print(f"   - Val:   {len(val_df)} ({len(val_df)/len(df_interact)*100:.1f}%)")
print(f"   - Test:  {len(test_df)} ({len(test_df)/len(df_interact)*100:.1f}%)")

# Kiểm tra không có user nào bị thiếu hoàn toàn khỏi train (tránh cold-start giả ở val/test)
users_in_val_test_only = (set(val_df['uid']) | set(test_df['uid'])) - set(train_df['uid'])
print(f"⚠️ Số user chỉ xuất hiện ở val/test mà không có trong train: {len(users_in_val_test_only)}")

# Lưu file Train/Val/Test
train_df[['uid', 'cid', 'label']].to_csv(
    os.path.join(PROCESSED_PATH, 'train.txt'), sep=' ', index=False, header=False
)
val_df[['uid', 'cid', 'label']].to_csv(
    os.path.join(PROCESSED_PATH, 'val.txt'), sep=' ', index=False, header=False
)
test_df[['uid', 'cid', 'label']].to_csv(
    os.path.join(PROCESSED_PATH, 'test.txt'), sep=' ', index=False, header=False
)
print("💾 Đã lưu train.txt, val.txt, test.txt")

📊 Tổng số dòng: 96579
   - Train: 74800 (77.4%)
   - Val:   9008 (9.3%)
   - Test:  12771 (13.2%)
⚠️ Số user chỉ xuất hiện ở val/test mà không có trong train: 0
💾 Đã lưu train.txt, val.txt, test.txt


In [ ]:
# 3. Xây dựng KG (CHỈ DÙNG TRAIN DATA cho cạnh User-Course)
# Lưu ý quan trọng: cạnh User-Course trong KG CHỈ được xây từ train_df.
# val_df và test_df TUYỆT ĐỐI không được đưa vào đây, nếu không agent RL
# sẽ "nhìn thấy" trước câu trả lời trong quá trình huấn luyện/lấy mẫu path.
triplets = []

# Quan hệ 0: User-Course (Lấy từ TRAIN_DF)
for _, row in train_df.iterrows():
    head = int(row['uid'])
    tail = int(row['cid']) + num_users
    triplets.append([head, 0, tail])

# Quan hệ 1: Course-Concept
cc_file = os.path.join(RELATIONS_PATH, 'course-concept.json')
df_cc = pd.read_csv(cc_file, sep='\t', header=None, names=['c', 'k'])
df_cc = df_cc[df_cc['c'].isin(valid_course_set)]

concept_map = {}
for _, row in df_cc.iterrows():
    if row['k'] not in concept_map: concept_map[row['k']] = len(concept_map)
    head = valid_course_map[row['c']] + num_users
    tail = concept_map[row['k']] + num_users + num_courses
    triplets.append([head, 1, tail])

# Quan hệ 2: Prerequisite
pre_file = os.path.join(RELATIONS_PATH, 'prerequisite-dependency.json')
df_pre = pd.read_csv(pre_file, sep='\t', header=None, names=['k1', 'k2'])
valid_k = set(concept_map.keys())
df_pre = df_pre[df_pre['k1'].isin(valid_k) & df_pre['k2'].isin(valid_k)]

offset_con = num_users + num_courses
for _, row in df_pre.iterrows():
    h = concept_map[row['k1']] + offset_con
    t = concept_map[row['k2']] + offset_con
    triplets.append([h, 2, t])

# Quan hệ 3: Teacher-Course
tc_file = os.path.join(RELATIONS_PATH, 'teacher-course.json')
df_tc = pd.read_csv(tc_file, sep='\t', header=None, names=['t', 'c'])
df_tc = df_tc[df_tc['c'].isin(valid_course_set)]

teacher_map = {}
offset_tea = offset_con + len(concept_map)
for _, row in df_tc.iterrows():
    if row['t'] not in teacher_map: teacher_map[row['t']] = len(teacher_map)
    h = teacher_map[row['t']] + offset_tea
    t = valid_course_map[row['c']] + num_users
    triplets.append([h, 3, t])

In [ ]:
# 4. TẠO CẠNH NGƯỢC (INVERSE RELATIONS) - QUAN TRỌNG!
inverse_triplets = []
for h, r, t in triplets:
    # Tạo cạnh ngược: Head <-> Tail, Relation ID mới = r + 4
    inverse_triplets.append([t, r + 4, h])

# Gộp cả 2 lại
all_triplets = triplets + inverse_triplets

In [ ]:
# 5. Lưu kết quả
kg_df = pd.DataFrame(all_triplets)
kg_df.to_csv(os.path.join(PROCESSED_PATH, 'kg_final.txt'), sep=' ', index=False, header=False)

# Cập nhật stats (relation_num vẫn là 8, không đổi)
with open(os.path.join(PROCESSED_PATH, 'entity_stats.txt'), 'w') as f:
    f.write(f"user_num={num_users}\n")
    f.write(f"course_num={num_courses}\n")
    f.write(f"concept_num={len(concept_map)}\n")
    f.write(f"teacher_num={len(teacher_map)}\n")
    f.write(f"relation_num=8\n")

print(f"Tổng cạnh: {len(all_triplets)} (Gốc: {len(triplets)})")
print("\n✅ Hoàn tất. File train.txt / val.txt / test.txt / kg_final.txt / entity_stats.txt đã sẵn sàng.")
print("👉 Bước tiếp theo: dùng val.txt trong quá trình huấn luyện RL (04_RL_Training_Eval.ipynb)")
print("   để chọn checkpoint tốt nhất; chỉ dùng test.txt MỘT LẦN ở bước đánh giá cuối cùng.")

Tổng cạnh: 435356 (Gốc: 217678)

✅ Hoàn tất. File train.txt / val.txt / test.txt / kg_final.txt / entity_stats.txt đã sẵn sàng.
👉 Bước tiếp theo: dùng val.txt trong quá trình huấn luyện RL (04_RL_Training_Eval.ipynb)
   để chọn checkpoint tốt nhất; chỉ dùng test.txt MỘT LẦN ở bước đánh giá cuối cùng.
